# Self-Correction & Error Recovery  (Reflection )



### Why This Matters

Code generation fails often. Without error handling, your agent is brittle and unusable.

```python
# Generated code:
result = df['sallary'].mean()  # typo!
# → KeyError: 'sallary'

# Without self-correction: Agent crashes
# With self-correction: Agent fixes and retries
```

### What You'll Build

An agent that:
1. Executes generated code
2. Catches exceptions
3. Sends error back to LLM
4. LLM analyzes error and fixes code
5. Retries (with max attempts limit)

### Key Concepts

- **Try-Catch Wrappers:** Safe code execution
- **Error Prompting:** Teaching LLM to read stack traces
- **Retry Logic:** Max attempts, exponential backoff
- **State Tracking:** Which errors occurred, how many retries

### Implementation Pattern

```python
def execute_with_retry(code, df, max_retries=3):
    for attempt in range(max_retries):
        try:
            result = run_code(code, df)
            return result
        except Exception as e:
            error_msg = f"Error: {str(e)}\nCode: {code}"
            fixed_code = ask_llm_to_fix(error_msg)
            code = fixed_code
    
    return "Failed after max retries"
```

### Learning Outcomes

- Error handling in AI systems
- Prompt engineering for debugging
- When to stop retrying (infinite loops)
- Logging failures for analysis

### Real-World Applications

- Jupyter notebook assistants
- SQL query generators
- API integration tools

---


In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

In [ ]:
!pip install bitsandbytes accelerate

In [ ]:

# -------------------------
# 0) Setup & Imports
# -------------------------
import json
import re
import torch
import pandas as pd
from typing import Dict, Any, Tuple, Optional, List

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# -------------------------
# 1) Load Model (same approach)
# -------------------------
# Change these paths to match your environment
model_path = "/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"
csv_path   = "/content/drive/MyDrive/hf_models/agent_project/employees.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model & tokenizer loaded")


In [ ]:
df = pd.read_csv(csv_path)

print("✅ Data Loaded:", df.shape)

In [ ]:
import ast
import signal
import torch
import pandas as pd
from typing import Dict, Any

def clean_llm_code(text: str) -> str:
    if "```" in text:
        parts = text.split("```")
        for part in parts:
            if "python" in part:
                return part.replace("python", "").strip()
        return parts[1].strip()

    valid = []
    for line in text.splitlines():
        if "=" in line or "df[" in line or "result" in line:
            valid.append(line)

    return "\n".join(valid)

In [ ]:
FORBIDDEN = (
    ast.Import, ast.ImportFrom, ast.With, ast.Try,
    ast.FunctionDef, ast.ClassDef
)

ALLOWED_BUILTINS = {
    "len": len, "min": min, "max": max, "sum": sum,
    "sorted": sorted, "round": round
}

# Methods that could expose raw data or perform dangerous operations
FORBIDDEN_METHODS = {
    # File export
    'to_csv', 'to_excel', 'to_json', 'to_sql',
    'to_clipboard', 'to_parquet', 'to_pickle',
    # Row-level exposure
    'head', 'tail', 'sample', 'iterrows',
    'itertuples', 'to_dict', 'to_records',
    # System access
    'system', 'popen', 'remove', 'rmdir',
}

def check_code(code: str):
    tree = ast.parse(code)

    for node in ast.walk(tree):
        if isinstance(node, FORBIDDEN):
            raise ValueError("Forbidden syntax detected.")

        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                if node.func.id in ["eval", "exec", "__import__", "open"]:
                    raise ValueError("Dangerous call blocked.")
        if isinstance(node, ast.Attribute):
            if node.attr in FORBIDDEN_METHODS:
                raise ValueError(f'Forbidden method: {node.attr}')


In [ ]:
def run_code(code: str, df: pd.DataFrame):
    """Execute pre-validated code in a restricted sandbox.
    
    NOTE: Caller is responsible for cleaning and checking code
    via clean_llm_code() and check_code() BEFORE calling this.
    """
    safe_globals = {
        "__builtins__": ALLOWED_BUILTINS,
        "pd": pd,
        "df": df
    }

    safe_locals: Dict[str, Any] = {}

    exec(code, safe_globals, safe_locals)

    if "result" not in safe_locals:
        raise ValueError("Code must assign result")

    return safe_locals["result"]


In [ ]:
def ask_llm(prompt: str, max_tokens: int = 300) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()

In [ ]:
def build_prompt(question: str, schema: dict) -> str:
    return f"""
You are a pandas data analyst.

Write executable pandas code to answer the question.

Rules:
- Use ONLY existing dataframe columns.
- No explanations.
- No markdown.
- No imports.
- Must assign final answer to variable: result.
- ONLY return aggregated results (sum, mean, count, max, min, groupby).
- NEVER return raw rows, individual records, or use head()/tail()/sample().
- If asked for 'who' or 'which person', return the COUNT or aggregated stat, not the row.

Schema:
{schema}

Question:
{question}
"""


In [ ]:
REPAIR_PROMPT = """
You are a pandas debugger.

Fix the code based on the error.

Rules:
- Return ONLY corrected python code.
- No explanations.
- No imports.
- Must assign result variable.
"""

In [ ]:
def ask_llm_to_fix(schema, question, code, error):
    prompt = f"""
{REPAIR_PROMPT}

Schema:
{schema}

Question:
{question}

Broken Code:
{code}

Error:
{error}

Return fixed code:
"""
    return ask_llm(prompt)

In [ ]:
def execute_with_retry(question, df, max_retries=10):

    schema = {
        "columns": list(df.columns),
        "rows": len(df)
    }

    code = ask_llm(build_prompt(question, schema))

    for attempt in range(max_retries):

        print(f"\n🔁 Attempt {attempt+1}")
        code_clean = clean_llm_code(code)
        print("Generated Code:\n", code_clean)

        try:
            check_code(code_clean)
            result = run_code(code_clean, df)
            print("✅ Execution Success")
            return result

        except Exception as e:
            print("❌ Failed:", e)

            if attempt == max_retries - 1:
                print("⛔ Max retries reached. Returning safe failure message.")
                return "I could not compute this safely. Try rephrasing."

            code = ask_llm_to_fix(schema, question, code, str(e))


In [ ]:
def solve(question: str, df: pd.DataFrame):
    return execute_with_retry(question, df)

In [ ]:
solve("What is the average salary in IT department?", df)

In [ ]:
solve("What is the average salary of employees with more than 5 years of experience?", df)


In [ ]:
solve("What is the average salary of employees with more than or equel 5 years of experience?", df)


In [ ]:
solve("Who has the highest salary among employees with more than 3 years of experience?", df)


In [ ]:
solve("What is the average sallary?", df)

In [ ]:
solve("What is the salary of the highest-paid employee in the department that has the largest number of employees?", df)